## Reading images with the ground truth 

### Pre- processing the ImHANDWRTITNG dataset 1 : metadata creation

In [3]:
import os
import csv
import concurrent.futures
from collections import defaultdict

# Paths
base_folder = "/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/"  # This should contain formsA-D, formsE-H, ...
lines_txt_path = "/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/ascii/lines.txt"  # Adjust if needed
output_csv = "Handwriting_1_matadata.csv"

# Step 1: Load and parse lines.txt
form_lines = defaultdict(list)

with open(lines_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        parts = line.strip().split()
        line_id = parts[0]
        form_id = "-".join(line_id.split("-")[:2])
        text = " ".join(parts[8:]).replace("|", " ")
        form_lines[form_id].append(text)

# Step 2: Scan all folders for image paths
def scan_for_images(folder):
    img_paths = {}
    for root, _, files in os.walk(folder):
        for file in files:
            if file.endswith((".png", ".jpeg", ".jpg")):
                form_id = os.path.splitext(file)[0]
                img_paths[form_id] = os.path.join(root, file)
    return img_paths

# Use multithreading for scanning
form_image_paths = {}

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(scan_for_images, os.path.join(base_folder, subfolder))
               for subfolder in os.listdir(base_folder)
               if subfolder.startswith("forms")]
    for future in concurrent.futures.as_completed(futures):
        form_image_paths.update(future.result())

# Step 3: Build dataset
dataset = []

for form_id, lines in form_lines.items():
    if form_id in form_image_paths:
        image_path = form_image_paths[form_id]
        text = " ".join(lines)
        dataset.append([form_id, image_path, text])

# Step 4: Write to CSV
with open(output_csv, "w", newline="", encoding="utf-8") as out_file:
    writer = csv.writer(out_file)
    writer.writerow(["form_id", "image_path", "transcription"])
    writer.writerows(dataset)

print(f"Dataset saved to: {output_csv}")

Dataset saved to: Handwriting_1_matadata.csv


## Preprocessing image

In [10]:
import os
import cv2
import numpy as np

def preprocess_images_in_folder(
    folder_path,
    out_folder,
    img_height=32,
    max_width=512,
    pad=True,
    binarize=False,
    keep_ext=".png"
):
    """
    Preprocess all images in a folder for OCR:
    - Resize to fixed height, keep aspect ratio
    - Optionally binarize
    - Pad or clip to max width
    - Normalize to [0, 1]
    - Save as uint8 grayscale images
    """
    os.makedirs(out_folder, exist_ok=True)

    for fname in os.listdir(folder_path):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            try:
                # Load grayscale image
                in_path = os.path.join(folder_path, fname)
                img = cv2.imread(in_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"⚠️ Skipped (not found): {fname}")
                    continue

                h, w = img.shape
                scale = img_height / h
                new_w = int(w * scale)
                img = cv2.resize(img, (new_w, img_height), interpolation=cv2.INTER_AREA)

                # Binarize if enabled
                if binarize:
                    img = cv2.adaptiveThreshold(
                        img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 10
                    )

                # Pad or clip to max_width
                if pad:
                    padded_img = np.ones((img_height, max_width), dtype=np.uint8) * 255
                    usable_width = min(new_w, max_width)
                    padded_img[:, :usable_width] = img[:, :usable_width]
                    img = padded_img
                else:
                    img = img[:, :max_width]

                # Save to output folder
                out_name = os.path.splitext(fname)[0] + keep_ext
                out_path = os.path.join(out_folder, out_name)
                cv2.imwrite(out_path, img)
                print(f"✅ Saved: {out_name}")

            except Exception as e:
                print(f"❌ Failed on {fname}: {e}")

In [17]:
preprocess_images_in_folder(
    folder_path="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D",
    out_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/formsA2D",
    img_height=48,
    max_width=1024,
    #keep_aspect_ratio = False,
    pad=True,
    binarize=False
)

✅ Saved: a04-006.png
✅ Saved: c03-087e.png
✅ Saved: a02-057.png
✅ Saved: d07-082.png
✅ Saved: d07-096.png
✅ Saved: a05-017.png
✅ Saved: b04-026.png
✅ Saved: d06-050.png
✅ Saved: c06-020.png
⚠️ Skipped (not found): d04-117 (1).png
✅ Saved: d05-030.png
✅ Saved: c02-000.png
✅ Saved: d04-021.png
✅ Saved: b06-042.png
✅ Saved: b06-056.png
✅ Saved: d04-008.png
✅ Saved: c04-044.png
⚠️ Skipped (not found): d04-032 (1).png
✅ Saved: c04-050.png
✅ Saved: d05-025.png
✅ Saved: b02-102.png
✅ Saved: a01-049x.png
✅ Saved: b04-147.png
✅ Saved: d06-086.png
✅ Saved: a03-047.png
⚠️ Skipped (not found): d03-112 (1).png
✅ Saved: a02-042.png
⚠️ Skipped (not found): d04-050 (1).png
✅ Saved: b03-098.png
✅ Saved: c03-087d.png
✅ Saved: c03-087f.png
✅ Saved: a04-039.png
✅ Saved: a01-007u.png
⚠️ Skipped (not found): d04-066 (1).png
✅ Saved: a05-000.png
✅ Saved: a06-100.png
✅ Saved: a06-114.png
✅ Saved: a06-128.png
✅ Saved: b01-000.png
✅ Saved: b01-014.png
✅ Saved: a01-020.png
✅ Saved: b06-097.png
✅ Saved: a01-011u.

## Language Detection

In [1]:
import fasttext
import torch

# Detect the best available device
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure correct path to model file
model_path = "/Users/krishnanand/Documents/Git/Dependencies/lid.176.bin"  # Update this path if necessary
model = fasttext.load_model(model_path)

def detect_language(text):
    """Detect language of given text using FastText."""
    predictions = model.predict(text)
    lang_code = predictions[0][0].replace('__label__', '')
    return lang_code

text = "Bonjour, comment allez-vous?"
detected_lang = detect_language(text)
print(f"Detected Language: {detected_lang}")

## Summarization  

In [2]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load summarization pipeline (T5-Base)
summarizer = pipeline("summarization", model="t5-base", device=device)

# Example input text
text = """
Artificial intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. 
These processes include learning, reasoning, and self-correction. AI is one of the most exciting and disruptive technologies 
of our time, influencing everything from healthcare to transportation.
"""

# Generate summary
summary = summarizer(text, max_length=100, min_length=30, do_sample=False)
summary_text = summary[0]['summary_text']
print("Summary in English:", summary_text)

# Load translation pipeline with a better model
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es", device=device)

# Translate the summary
translated_summary = translator(summary_text)
print("Translated Summary in Spanish:", translated_summary[0]['translation_text'])

## Translator

In [3]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load the NLLB model for French to Hindi translation
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M", device=device)

# Example French text
text = "Bonjour, comment allez-vous aujourd'hui?"

# Translate the text from French to English
translated_text = translator(text, src_lang="fra_Latn", tgt_lang="eng_Latn")
english_translation = translated_text[0]['translation_text']
print("Translated to English:", english_translation)

# Now, translate from English to Hindi
translated_to_hindi = translator(english_translation, src_lang="eng_Latn", tgt_lang="hin_Deva")
print("Translated to Hindi:", translated_to_hindi[0]['translation_text'])

## TTS

In [4]:
from TTS.api import TTS
import torch

# Detect best available device (CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Load a single-language English TTS model and move it to the selected device
tts = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(device)

# Text to convert
text = ("The error 'Model is not multi-lingual but language is provided' means that "
        "the model does not support multiple languages. This model can only generate speech in English, "
        "so specifying language (French) is invalid.")

# Convert text to speech and save as an audio file
tts.tts_to_file(text=text, file_path="/Users/krishnanand/Documents/Git/Tests/test123.wav")

print("Speech synthesis complete! Output saved as 'test123.wav'.")

In [1]:
pip install colabcode

In [3]:
%pip install colabcode